In [0]:
WITH params AS (
  SELECT
    30 AS p_window_days,
    CAST('7405606764056809' AS STRING) AS p_workspace_id,
    'azure-databricks-3dt-2nd-5th' AS p_workspace_name,
    'https://adb-7405606764056809.9.azuredatabricks.net' AS p_workspace_url
),
usage_scoped AS (
  SELECT
    u.usage_date,
    u.usage_start_time,
    u.usage_end_time,
    CAST(u.workspace_id AS STRING) AS workspace_id,
    u.sku_name,
    u.cloud,
    u.usage_unit,
    u.usage_quantity,
    u.billing_origin_product,
    p.p_workspace_name AS workspace_name,
    p.p_workspace_url AS workspace_url
  FROM system.billing.usage u
  CROSS JOIN params p
  WHERE CAST(u.workspace_id AS STRING) = p.p_workspace_id
    AND u.usage_date >= date_sub(current_date(), p.p_window_days)
),
usage_with_prices AS (
  SELECT
    u.usage_date,
    u.usage_start_time,
    u.usage_end_time,
    u.workspace_id,
    u.sku_name,
    u.cloud,
    u.usage_unit,
    CAST(u.usage_quantity AS DOUBLE) AS usage_quantity,
    u.billing_origin_product,
    u.workspace_name,
    u.workspace_url,
    CAST(COALESCE(lp.pricing.effective_list.default, lp.pricing.default) AS DOUBLE) AS unit_list_price,
    COALESCE(lp.currency_code, 'USD') AS currency_code
  FROM usage_scoped u
  LEFT JOIN system.billing.list_prices lp
    ON u.cloud = lp.cloud
   AND u.sku_name = lp.sku_name
   AND u.usage_unit = lp.usage_unit
   AND u.usage_end_time >= lp.price_start_time
   AND (lp.price_end_time IS NULL OR u.usage_end_time < lp.price_end_time)
),
daily_cost AS (
  SELECT
    workspace_id,
    workspace_name,
    workspace_url,
    usage_date AS usage_date_utc,
    date(from_utc_timestamp(CAST(usage_date AS TIMESTAMP), 'Asia/Seoul')) AS usage_date_kst,
    ROUND(SUM(usage_quantity * COALESCE(unit_list_price, 0.0)), 4) AS daily_cost_usd,
    ROUND(SUM(usage_quantity), 4) AS daily_usage_quantity,
    MAX(currency_code) AS currency_code
  FROM usage_with_prices
  GROUP BY
    workspace_id,
    workspace_name,
    workspace_url,
    usage_date,
    date(from_utc_timestamp(CAST(usage_date AS TIMESTAMP), 'Asia/Seoul'))
),
finalized AS (
  SELECT
    workspace_id,
    workspace_name,
    workspace_url,
    usage_date_utc,
    usage_date_kst,
    daily_cost_usd,
    ROUND(
      SUM(daily_cost_usd) OVER (
        PARTITION BY workspace_id
        ORDER BY usage_date_utc
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
      ),
      4
    ) AS running_cost_usd,
    ROUND(SUM(daily_cost_usd) OVER (PARTITION BY workspace_id), 4) AS total_cost_usd_30d,
    daily_usage_quantity,
    currency_code
  FROM daily_cost
)
SELECT
  workspace_id,
  workspace_name,
  workspace_url,
  usage_date_utc,
  usage_date_kst,
  CAST(usage_date_utc AS TIMESTAMP) AS usage_time_utc,
  from_utc_timestamp(CAST(usage_date_utc AS TIMESTAMP), 'Asia/Seoul') AS usage_time_kst,
  daily_cost_usd,
  running_cost_usd,
  total_cost_usd_30d,
  daily_usage_quantity,
  currency_code
FROM finalized
ORDER BY usage_date_utc;
